# Evie-Amplify (EA) component ablation, all three domains (AV vision, AF finance, AL language), one notebook

Turn on **GPU (T4 x2)** and **Internet**, then *Run All*. Nothing else to do.

**What is ablated.** EA multiplies the AdamW step by `m_0(x)/sqrt(w)`, `x = gamma0*sigma2`, `m_0(x)=sqrt((1+x)/(1+2x))`, harmonic `gamma_t`,
`gamma0` calibrated once at step 125 (before that `gamma0 = 1`, the "warm-up gate"). Each arm removes exactly one ingredient:

| arm | what changed vs EA | LR | tests |
|---|---|---|---|
| AdamW | everything off | tuned | baseline |
| Evie | shrink-only gate (no 1/sqrt(w), different gate form) | tuned | Evie -> EA step |
| **EvieAmplify (EA)** | full method | tuned | |
| EA_noRecenter | no 1/sqrt(w) factor (gate only, `m_0(x)`) | tuned | **scale** effect |
| M1 | constant 1/sqrt(w), no gate | EA's | **gate** effect at fixed scale |
| EA_noWarmup | `gamma0 = 0` until calibration | EA's | **warm-up gate** |
| EA_fixedG0 | `gamma0 = 1`, never calibrated | EA's | **calibration** |
| EA_fixedSched | warm-up off + constant `gamma` (no harmonic schedule) | EA's | **harmonic schedule** (contrast with EA_noWarmup) |

Protocol: quarter-decade LR grid + edge extension + refinement for every tuned arm (validation loss only, disjoint tuning seeds/units),
one shared run per arm/unit/seed, large-batch evaluation, exact Wilcoxon + Holm over the six declared contrasts. The clamps (floor 1e-6, cap 8) are
inactive by theorem B2; their occupancy is reported (`floor_frac`, `cap_frac`) and must be 0.

**Time budget / resume.** Everything stops cleanly after 9 h 55 min (`BUDGET_MIN`) and writes `/kaggle/working/results_pack.zip`.
To continue: download that zip, start a new session, *Add Data* -> upload it (or attach this notebook's previous Output), turn on GPU + Internet, *Run All* again.
The cells run one after another (AV, AF, AL) on both GPUs, sharing the one 9 h 55 min budget. The log says `RESUME: started from checkpoint ...` and finished runs are skipped.

In [ ]:
# ---- 1. install the embedded code (nothing to upload or clone) -------------------------------------------------
import base64, io, os, sys, subprocess, tarfile, time, glob, json, threading
BUNDLE = """
...
"""
ON_KAGGLE = os.path.isdir("/kaggle/working")
ROOT = "/kaggle/working/repo" if ON_KAGGLE else os.path.abspath("./repo_nb")
os.makedirs(ROOT, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as _t:
    try:
        _t.extractall(ROOT, filter="data")
    except TypeError:                      # Python < 3.12
        _t.extractall(ROOT)
os.chdir(ROOT)
sys.path.insert(0, ROOT)
if ON_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "yfinance", "datasets"], check=False)

def sh(cmd, env=None, prefix=""):
    """Run a shell command, stream its output live, return (returncode, captured_text)."""
    e = dict(os.environ); e.update({k: str(v) for k, v in (env or {}).items()})
    if cmd.startswith("python "):
        cmd = '"' + sys.executable + '" ' + cmd[len("python "):]
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=ROOT)
    buf = []
    for line in p.stdout:
        print(prefix + line, end="", flush=True); buf.append(line)
    p.wait()
    return p.returncode, "".join(buf)

try:
    import torch
    N_GPU = torch.cuda.device_count()
    print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])
except Exception as _e:
    N_GPU = 0; print("torch not importable:", _e)
if ON_KAGGLE and N_GPU == 0:
    raise SystemExit("No GPU found: turn on 'Accelerator: GPU T4 x2' in the notebook settings and run again.")
if ON_KAGGLE:
    import urllib.request
    try:
        urllib.request.urlopen("https://pypi.org", timeout=10)
    except Exception:
        raise SystemExit("No internet: turn on 'Internet' in the notebook settings and run again.")

In [ ]:
# ---- 2. settings (defaults are the plan; change only if you know why) -------------------------------------------
CELLS = ["AV", "AF", "AL"]
PROFILE = "ea"
BUDGET_MIN = 595            # 9 h 55 min for ALL three cells together
ELAPSED_OFFSET_MIN = 0      # minutes of this Kaggle session already used before running this notebook
SMOKE = False               # developer switch: tiny fake-data run, no downloads
GPUS = max(1, min(2, N_GPU)) if not SMOKE else 1
ENV = {"EVIE_DATA": "/kaggle/working/data" if ON_KAGGLE else os.path.abspath("./data_nb"),
       "EVIE_RESULTS": "/kaggle/working/results" if ON_KAGGLE else os.path.abspath("./results_nb"),
       "EVIE_PACK": ("/kaggle/working/" if ON_KAGGLE else os.path.abspath("./")) + "/results_pack.zip"}
os.environ.update(ENV)
T0 = time.time()
print(ENV, "GPUS =", GPUS)

In [ ]:
# ---- 3. data for all three domains (CIFAR-100 ~1 min, tickers ~5-10 min, WikiText-2 ~2 min; skipped if cached / restored) ----
old = sorted(glob.glob("/kaggle/input/**/run1a_checkpoint.json", recursive=True))
if old:
    print("finance: using the ticker list of the existing runs:", old[0])
    extra = f"--tickers-from {old[0]}"
else:
    extra = ""
    print("[WARN] finance: no run1a_checkpoint.json under /kaggle/input: tickers are recomputed with the legacy rule (seed 1234, first 100 that download). "
          "To use exactly the old list, attach the dataset containing run1a_checkpoint.json and rerun.")
if not SMOKE:
    for dom, ex in (("vision", ""), ("finance", extra), ("language", "")):
        rc, _ = sh(f"python -m evie_exp.scripts.prepare_data --domain {dom} {ex}")
        assert rc == 0, f"data preparation failed for {dom} (is Internet on?)"

In [ ]:
# ---- 4. measure this machine per cell (about 2-5 min each) and print the projected hours --------------------------------
sm = " --smoke" if SMOKE else ""
for CELL in CELLS:
    sh(f"python -m evie_exp.scripts.time_probe --cell {CELL} --profile {PROFILE} --gpus {GPUS}{sm}")

In [ ]:
# ---- 5. the three cells in turn: tune LRs -> select -> evaluation of every arm (both GPUs in parallel). Resumable. ----------
sm = " --smoke --max-units 2" if SMOKE else ""
STATUS = {}
for CELL in CELLS:
    used = ELAPSED_OFFSET_MIN + (time.time() - T0) / 60.0
    if BUDGET_MIN - used < 3:
        STATUS[CELL] = "not started (budget used up)"; continue
    rc, out = sh(f"python -m evie_exp.scripts.run_all --cell {CELL} --profile {PROFILE} --gpus {GPUS} "
                 f"--time-budget-min {BUDGET_MIN} --elapsed-offset-min {used}{sm}", prefix=f"[{CELL}] ")
    STATUS[CELL] = "complete" if "ALL STEPS COMPLETE" in out else f"incomplete (exit {rc})"
    if rc not in (0, 10):
        print(f"[{CELL}] exited with {rc}; continuing with the next cell, rerun later to retry this one")
print("\nSTATUS:", STATUS)
print("ALL DONE" if all(v == "complete" for v in STATUS.values()) else
      "NOT COMPLETE (budget or an error): download /kaggle/working/results_pack.zip, attach it to the next session and Run All again.")

In [ ]:
# ---- 6. tables per cell (paired differences, exact Wilcoxon, Holm, CIs, diagnostics) --------------------------------------
import shutil
for CELL in CELLS:
    sm = " --fake-data --max-units 2" if SMOKE else ""
    sh(f"python -m evie_exp.analysis.build_tables --cell {CELL} --profile {PROFILE} --results {ENV['EVIE_RESULTS']} --out {ROOT}/tables{sm}")
if ON_KAGGLE and os.path.isdir(f"{ROOT}/tables"):
    shutil.copytree(f"{ROOT}/tables", "/kaggle/working/tables", dirs_exist_ok=True)
print("\nFiles to keep: results_pack.zip (all run records + logs), tables/*.md|json")